# Phase 3: Neural ODE Model Development

**Project**: Neural PK-PD Modeling with Physics-Informed Neural ODEs  
**Date**: February 24, 2026  
**Previous Phase**: [phase1_2_data_exploration.ipynb](phase1_2_data_exploration.ipynb)  

---

## 🎯 Phase 3 Objectives

1. **Multi-Task Neural Architecture** - Shared encoder with task-specific heads
2. **Neural ODE Integration** - Learn PK dynamics using `torchdiffeq`
3. **Physics-Informed Loss** - Incorporate mechanistic constraints
4. **4 Prediction Tasks**:
   - Binding affinity (regression)
   - hERG inhibition (classification)
   - Caco-2 permeability (classification)
   - Hepatocyte clearance (regression)

### Target Metrics

| Task | Metric | Target |
|------|--------|--------|
| Binding Affinity | R² | > 0.6 |
| hERG Inhibition | AUROC | > 0.8 |
| Caco-2 Permeability | AUROC | > 0.75 |
| Clearance | RMSE | < 1.0 |

---
## 1. Environment Setup & Imports

In [ ]:
# Core Libraries
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Deep Learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

# Neural ODE
from torchdiffeq import odeint

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, r2_score, mean_absolute_error,
    roc_auc_score, accuracy_score, f1_score, classification_report
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

In [ ]:

# =============================================================================
# CELL: Execution Timestamp Logger
# PURPOSE: Record notebook start time, environment info, and capture cell-level
#          execution timestamps for a full session timeline
# INPUTS:  None
# OUTPUTS: Prints session header; provides _ts() and log_elapsed() helpers
# =============================================================================
import datetime
import platform
import sys

# ── Execution log (ordered dict: cell_name → datetime) ──────────────────────
_execution_log = {}   # {cell_name: datetime}

def _ts(cell_name):
    """Log and print timestamp when a cell starts executing."""
    now = datetime.datetime.now()
    _execution_log[cell_name] = now
    seq = list(_execution_log.keys()).index(cell_name) + 1
    if seq == 1:
        elapsed = "(start)"
    else:
        first = list(_execution_log.values())[0]
        elapsed = f"+{(now - first).total_seconds():.1f}s"
    print(f"⏱️  [{now.strftime('%H:%M:%S')}]  #{seq:02d}  {elapsed:<10}  {cell_name}")

# ── Session start time ────────────────────────────────────────────────────────
SESSION_START = datetime.datetime.now()

def log_elapsed(label: str = "Checkpoint") -> None:
    """Print elapsed time since SESSION_START. Call after any major cell."""
    delta = datetime.datetime.now() - SESSION_START
    total_sec = int(delta.total_seconds())
    h, rem = divmod(total_sec, 3600)
    m, s = divmod(rem, 60)
    print(f"  ⏱  {label}: {h:02d}h {m:02d}m {s:02d}s elapsed")

# ── Session header ────────────────────────────────────────────────────────────
print("=" * 65)
print("  PHASE 3 — Neural ODE PK-PD Model")
print("  Execution Timestamp Logger")
print("=" * 65)
print(f"  Started     : {SESSION_START.strftime('%Y-%m-%d  %H:%M:%S')}")
print(f"  Python      : {sys.version.split()[0]}  ({platform.python_implementation()})")
print(f"  Platform    : {platform.system()} {platform.release()} ({platform.machine()})")

import torch
print(f"  PyTorch     : {torch.__version__}")
device_name = (
    torch.cuda.get_device_name(0) if torch.cuda.is_available()
    else ("MPS" if torch.backends.mps.is_available() else "CPU (no GPU)")
)
print(f"  Device      : {device_name}")

try:
    import rdkit
    print(f"  RDKit       : {rdkit.__version__}")
except ImportError:
    print("  RDKit       : NOT FOUND")

try:
    import torchdiffeq
    print(f"  torchdiffeq : {torchdiffeq.__version__}")
except (ImportError, AttributeError):
    print("  torchdiffeq : installed (version attr unavailable)")

print("=" * 65)
print("  Notebook: phase3_neural_ode_model.ipynb")
print("  Project : Neural PK-PD Modeling with ODE")
print("=" * 65)
print("✅ Timestamp logger ready  →  _ts('cell name') is active")
print("   Run the timeline summary cell at the end to view the full log.")


---
## 2. Configuration & Hyperparameters

In [ ]:

# ============================================
# HYPERPARAMETERS
# ============================================
_ts("Cell 02 - Configuration")

config = {
    # Data
    'test_size': 0.2,
    'val_size': 0.1,

    # Model Architecture
    'input_dim': 68,          # 4 physico + 64 Morgan ECFP4
    'hidden_dim': 128,
    'latent_dim': 64,
    'dropout': 0.2,

    # Training  (more epochs needed since each epoch does fewer steps now)
    'batch_size': 64,
    'epochs': 300,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'patience': 40,

    'grad_clip': 1.0,

    # Multi-task Loss Weights
    'w_binding': 1.0,
    'w_herg': 1.0,
    'w_caco2': 1.0,
    'w_clearance': 1.0,
    'w_physics': 0.1,

    'herg_pos_weight': 2.5,
}

print("Configuration loaded:")
for k, v in config.items():
    print(f"  {k}: {v}")


---
## 3. Data Loading

Load preprocessed data from Phase 1-2. See [phase1_2_data_exploration.ipynb](phase1_2_data_exploration.ipynb) for data processing details.

In [ ]:

# ============================================
# LOAD RAW DATASETS
# ============================================
_ts("Cell 03 - Dataset Loading")

# Paths
DATA_DIR = 'data/raw/'

# Load ChEMBL binding affinity data
chembl_df = pd.read_csv(f'{DATA_DIR}chembl/chembl_binding_affinity.csv')
print(f"ChEMBL: {len(chembl_df)} records")

# Load TDC ADMET datasets
herg_df = pd.read_csv(f'{DATA_DIR}tdc/tdc_herg.csv')
caco2_df = pd.read_csv(f'{DATA_DIR}tdc/tdc_caco2_wang.csv')
clearance_df = pd.read_csv(f'{DATA_DIR}tdc/tdc_clearance_hepatocyte_az.csv')

print(f"hERG: {len(herg_df)} records")
print(f"Caco-2: {len(caco2_df)} records")
print(f"Clearance: {len(clearance_df)} records")
print(f"\nTotal: {len(chembl_df) + len(herg_df) + len(caco2_df) + len(clearance_df)} records")


In [ ]:

# ============================================
# INSPECT DATA STRUCTURE
# ============================================
_ts("Cell 04 - Dataset Inspection")

print("ChEMBL columns:", chembl_df.columns.tolist())
print("\nhERG columns:", herg_df.columns.tolist())
print("\nCaco-2 columns:", caco2_df.columns.tolist())
print("\nClearance columns:", clearance_df.columns.tolist())


---
## 4. Feature Engineering & Preprocessing

In [ ]:

# ============================================
# PREPARE FEATURES FOR EACH TASK
# ============================================
_ts("Cell 05 - Task Data Prep Fn")

def prepare_task_data(df, feature_cols, target_col, task_type='regression'):
    """
    Prepare features and targets for a specific task.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe
    feature_cols : list
        Column names for features
    target_col : str
        Column name for target
    task_type : str
        'regression' or 'classification'
        
    Returns:
    --------
    X, y : numpy arrays
    """
    # Drop rows with missing values
    subset_cols = feature_cols + [target_col]
    df_clean = df.dropna(subset=subset_cols)
    
    X = df_clean[feature_cols].values
    y = df_clean[target_col].values
    
    print(f"  Samples: {len(X)}, Features: {X.shape[1]}, Target: {target_col}")
    
    return X, y

print("Feature preparation function defined.")


In [ ]:

# ============================================
# MORGAN FINGERPRINT GENERATION (RDKit)
# ============================================
_ts("Cell 06 - Morgan Fingerprints")

from rdkit import Chem
from rdkit.Chem import AllChem

N_BITS  = 64     # Morgan fingerprint bits
RADIUS  = 2      # ECFP4 (radius=2)

def smiles_to_morgan(smiles_series, n_bits=N_BITS, radius=RADIUS):
    """Convert SMILES column to Morgan fingerprint array (n_mols x n_bits)."""
    fps = []
    for smi in smiles_series:
        try:
            mol = Chem.MolFromSmiles(str(smi))
            if mol is not None:
                fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
                fps.append(list(fp))
            else:
                fps.append([0] * n_bits)
        except Exception:
            fps.append([0] * n_bits)
    return np.array(fps, dtype=np.float32)

# Generate fingerprints for datasets that have SMILES
print("Generating Morgan fingerprints (ECFP4, 64-bit)...")
fp_herg      = smiles_to_morgan(herg_df['smiles'])
fp_caco2     = smiles_to_morgan(caco2_df['smiles'])
fp_clearance = smiles_to_morgan(clearance_df['smiles'])

print(f"  hERG fingerprints:      {fp_herg.shape}")
print(f"  Caco-2 fingerprints:    {fp_caco2.shape}")
print(f"  Clearance fingerprints: {fp_clearance.shape}")
print(f"\nFinal feature dim per task: 4 physico + {N_BITS} Morgan = {4 + N_BITS}")


In [ ]:

# ============================================
# EXTRACT FEATURES PER TASK
# ============================================
_ts("Cell 07 - Feature Extraction")

PHYSICO_FEATURES = ['MW', 'LogP', 'HBA', 'HBD']   # 4 shared physico descriptors
FEAT_DIM = len(PHYSICO_FEATURES) + N_BITS           # 4 + 64 = 68

print(f"Feature dimensions: {len(PHYSICO_FEATURES)} physico + {N_BITS} Morgan = {FEAT_DIM}\n")

# ── Task 1: Binding Affinity (ChEMBL) — regression ──────────────────
# ChEMBL has no SMILES → zero-pad the Morgan section
X_physico, y_binding = prepare_task_data(chembl_df, PHYSICO_FEATURES, 'pchembl_value')
fp_chembl = np.zeros((len(X_physico), N_BITS), dtype=np.float32)
X_binding = np.hstack([X_physico, fp_chembl])
print(f"Task 1 Binding   — samples: {len(X_binding)}, features: {X_binding.shape[1]}")

# ── Task 2: hERG Inhibition (TDC) — classification ──────────────────
X_physico, y_herg = prepare_task_data(herg_df, PHYSICO_FEATURES, 'herg')
X_herg = np.hstack([X_physico, fp_herg[:len(X_physico)]])
print(f"Task 2 hERG      — samples: {len(X_herg)}, features: {X_herg.shape[1]}, "
      f"pos%: {y_herg.mean():.1%}")

# ── Task 3: Caco-2 Permeability (TDC) — regression ──────────────────
X_physico, y_caco2 = prepare_task_data(caco2_df, PHYSICO_FEATURES, 'caco2_wang')
X_caco2 = np.hstack([X_physico, fp_caco2[:len(X_physico)]])
print(f"Task 3 Caco-2    — samples: {len(X_caco2)}, features: {X_caco2.shape[1]}")

# ── Task 4: Hepatocyte Clearance (TDC) — regression ─────────────────
clearance_padded = clearance_df.copy()
clearance_padded['HBA'] = 0
clearance_padded['HBD'] = 0
X_physico, y_clearance = prepare_task_data(clearance_padded, PHYSICO_FEATURES, 'clearance_hepatocyte_az')
X_clearance = np.hstack([X_physico, fp_clearance[:len(X_physico)]])
print(f"Task 4 Clearance — samples: {len(X_clearance)}, features: {X_clearance.shape[1]}")

print("\nAll tasks prepared.")


In [ ]:

# ============================================
# BUILD TRAIN / VAL / TEST DATA LOADERS
# ============================================
_ts("Cell 08 - DataLoaders")

from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def make_loaders(X, y, config, is_regression=True, random_state=42):
    """
    Split into train/val/test and return DataLoaders with {'X', 'y'} batches.
    Normalizes both features AND targets (for regression) on the train split only.
    """
    X_tv, X_test, y_tv, y_test = train_test_split(
        X, y, test_size=config['test_size'], random_state=random_state
    )
    val_frac = config['val_size'] / (1 - config['test_size'])
    X_train, X_val, y_train, y_val = train_test_split(
        X_tv, y_tv, test_size=val_frac, random_state=random_state
    )

    # Normalize features (fit on train only)
    feat_scaler = StandardScaler()
    X_train = feat_scaler.fit_transform(X_train)
    X_val   = feat_scaler.transform(X_val)
    X_test  = feat_scaler.transform(X_test)

    # Normalize regression targets (makes MSE comparable across tasks)
    target_scaler = None
    if is_regression:
        target_scaler = StandardScaler()
        y_train = target_scaler.fit_transform(y_train.reshape(-1, 1)).flatten()
        y_val   = target_scaler.transform(y_val.reshape(-1, 1)).flatten()
        y_test  = target_scaler.transform(y_test.reshape(-1, 1)).flatten()

    def to_loader(Xa, ya, shuffle=True):
        ds = TensorDataset(
            torch.FloatTensor(Xa),
            torch.FloatTensor(ya).unsqueeze(1)
        )
        def collate(batch):
            Xb, yb = zip(*batch)
            return {'X': torch.stack(Xb), 'y': torch.stack(yb)}
        return DataLoader(ds, batch_size=config['batch_size'],
                          shuffle=shuffle, collate_fn=collate)

    return (
        to_loader(X_train, y_train, shuffle=True),
        to_loader(X_val,   y_val,   shuffle=False),
        to_loader(X_test,  y_test,  shuffle=False),
        feat_scaler,
        target_scaler,
    )

# Regression / classification flags per task
task_types = {
    'binding':   True,   # regression
    'herg':      False,  # classification
    'caco2':     True,   # regression
    'clearance': True,   # regression
}

tasks_raw = {
    'binding':   (X_binding,   y_binding),
    'herg':      (X_herg,      y_herg),
    'caco2':     (X_caco2,     y_caco2),
    'clearance': (X_clearance, y_clearance),
}

train_loaders, val_loaders, test_loaders = {}, {}, {}
feat_scalers, target_scalers = {}, {}

for task, (X, y) in tasks_raw.items():
    tr, va, te, fs, ts = make_loaders(X, y, config, is_regression=task_types[task])
    train_loaders[task]   = tr
    val_loaders[task]     = va
    test_loaders[task]    = te
    feat_scalers[task]    = fs
    target_scalers[task]  = ts   # None for classification

print("DataLoaders created (regression targets normalized to mean=0, std=1):")
for task, loader in train_loaders.items():
    n = len(loader.dataset)
    reg = "regression" if task_types[task] else "classification"
    print(f"  {task:<12}  {reg:<14}  train={n}, "
          f"val={len(val_loaders[task].dataset)}, "
          f"test={len(test_loaders[task].dataset)}")


---
## 5. Multi-Task Dataset Class

In [ ]:

# ============================================
# MULTI-TASK DATASET
# ============================================
_ts("Cell 09 - MultiTaskDataset")

class MultiTaskDataset(Dataset):
    """
    Dataset for multi-task learning with different sample sizes per task.
    """
    def __init__(self, task_data_dict):
        """
        Parameters:
        -----------
        task_data_dict : dict
            Dictionary with task names as keys, (X, y) tuples as values
        """
        self.tasks = list(task_data_dict.keys())
        self.data = task_data_dict
        
        # Total samples across all tasks
        self.total_samples = sum(len(v[0]) for v in task_data_dict.values())
        
        # Create index mapping: global_idx -> (task, local_idx)
        self.index_map = []
        for task in self.tasks:
            n_samples = len(self.data[task][0])
            for i in range(n_samples):
                self.index_map.append((task, i))
    
    def __len__(self):
        return self.total_samples
    
    def __getitem__(self, idx):
        task, local_idx = self.index_map[idx]
        X, y = self.data[task]
        return {
            'features': torch.FloatTensor(X[local_idx]),
            'target': torch.FloatTensor([y[local_idx]]),
            'task': task
        }

print("MultiTaskDataset class defined.")


---
## 6. Neural Network Architecture

### 6.1 Shared Encoder

In [ ]:

# ============================================
# SHARED ENCODER  (LayerNorm — safe for multi-task interleaved training)
# ============================================
_ts("Cell 10 - Shared Encoder")

class SharedEncoder(nn.Module):
    """
    Shared feature encoder for all tasks.
    Uses LayerNorm instead of BatchNorm to avoid running-statistics
    corruption caused by interleaved multi-task batches.
    """
    def __init__(self, input_dim, hidden_dim, latent_dim, dropout=0.3):
        super(SharedEncoder, self).__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.encoder(x)

print("SharedEncoder defined (LayerNorm, safe for interleaved multi-task training).")


### 6.2 Task-Specific Heads

In [ ]:

# ============================================
# TASK-SPECIFIC HEADS
# ============================================
_ts("Cell 11 - Task Heads")

class RegressionHead(nn.Module):
    """Head for regression tasks (binding affinity, clearance)"""
    def __init__(self, latent_dim):
        super(RegressionHead, self).__init__()
        self.head = nn.Sequential(
            nn.Linear(latent_dim, latent_dim // 2),
            nn.ReLU(),
            nn.Linear(latent_dim // 2, 1)
        )
    
    def forward(self, x):
        return self.head(x)


class ClassificationHead(nn.Module):
    """Head for binary classification tasks (hERG, Caco-2)"""
    def __init__(self, latent_dim):
        super(ClassificationHead, self).__init__()
        self.head = nn.Sequential(
            nn.Linear(latent_dim, latent_dim // 2),
            nn.ReLU(),
            nn.Linear(latent_dim // 2, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.head(x)

print("Task heads defined: RegressionHead, ClassificationHead")


### 6.3 Neural ODE Component

In [ ]:

# ============================================
# NEURAL ODE FOR PK DYNAMICS
# ============================================
_ts("Cell 12 - PKODEFunc")

class PKODEFunc(nn.Module):
    """
    Neural ODE function for pharmacokinetic dynamics.
    
    Models: dC/dt = f(C, molecular_features)
    
    Can learn clearance and volume parameters from molecular structure.
    """
    def __init__(self, latent_dim):
        super(PKODEFunc, self).__init__()
        
        # Network to predict PK parameters from latent features
        self.pk_params = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.Tanh(),
            nn.Linear(32, 2)  # [CL, V] predicted
        )
        
        # Latent features (set during forward pass)
        self.latent = None
    
    def set_latent(self, latent):
        """Set molecular latent features for ODE integration"""
        self.latent = latent
    
    def forward(self, t, C):
        """
        ODE dynamics: dC/dt = -k * C where k = CL/V
        
        Parameters:
        -----------
        t : tensor
            Time point
        C : tensor
            Concentration (batch_size, 1)
        """
        if self.latent is None:
            raise ValueError("Latent features not set. Call set_latent() first.")
        
        # Get PK parameters
        pk = self.pk_params(self.latent)
        CL = torch.exp(pk[:, 0:1])  # Ensure positive clearance
        V = torch.exp(pk[:, 1:2])   # Ensure positive volume
        k = CL / V
        
        # First-order elimination: dC/dt = -k * C
        dCdt = -k * C
        
        return dCdt

print("PKODEFunc defined for neural ODE PK modeling.")


### 6.4 Complete Multi-Task Model

In [ ]:

# ============================================
# MULTI-TASK PK-PD MODEL
# ============================================
_ts("Cell 13 - Full Model")

class MultiTaskPKPDModel(nn.Module):
    """
    Complete multi-task model for PK-PD predictions.

    Architecture:
    - Shared encoder: molecular features → latent space
    - Task heads: latent → task-specific predictions
      * binding   → RegressionHead     (pIC50)
      * hERG      → ClassificationHead (binary, sigmoid output)
      * caco2     → RegressionHead     (log-permeability, continuous)
      * clearance → RegressionHead     (clearance value)
    - Neural ODE: latent → PK time-course (optional)
    """
    def __init__(self, config):
        super(MultiTaskPKPDModel, self).__init__()

        input_dim  = config['input_dim']
        hidden_dim = config['hidden_dim']
        latent_dim = config['latent_dim']
        dropout    = config['dropout']

        # Shared encoder
        self.encoder = SharedEncoder(input_dim, hidden_dim, latent_dim, dropout)

        # Task-specific heads
        self.binding_head   = RegressionHead(latent_dim)      # pIC50 regression
        self.herg_head      = ClassificationHead(latent_dim)  # hERG binary classification
        self.caco2_head     = RegressionHead(latent_dim)      # Caco-2 regression (log-perm)
        self.clearance_head = RegressionHead(latent_dim)      # Clearance regression

        # Neural ODE for PK dynamics
        self.ode_func = PKODEFunc(latent_dim)

    def forward(self, x, task=None):
        latent = self.encoder(x)

        if task == 'binding':
            return self.binding_head(latent)
        elif task == 'herg':
            return self.herg_head(latent)
        elif task == 'caco2':
            return self.caco2_head(latent)
        elif task == 'clearance':
            return self.clearance_head(latent)
        elif task == 'all':
            return {
                'binding':   self.binding_head(latent),
                'herg':      self.herg_head(latent),
                'caco2':     self.caco2_head(latent),
                'clearance': self.clearance_head(latent),
                'latent':    latent,
            }
        else:
            return latent

    def predict_pk_curve(self, x, t_span, C0=1.0):
        latent = self.encoder(x)
        self.ode_func.set_latent(latent)
        C_init = torch.full((x.shape[0], 1), C0, device=x.device)
        C_t = odeint(self.ode_func, C_init, t_span)
        return C_t  # (time_points, batch_size, 1)

print("MultiTaskPKPDModel defined.")
print("\nModel components:")
print("  - SharedEncoder: features → latent")
print("  - RegressionHead:     binding, caco2, clearance")
print("  - ClassificationHead: hERG")
print("  - PKODEFunc:          Neural ODE for PK curves")


In [ ]:

# ============================================
# INSTANTIATE MODEL
# ============================================
_ts("Cell 14 - Instantiate Model")

model = MultiTaskPKPDModel(config).to(device)

# Print model summary
print("Model Architecture:")
print("=" * 50)
print(model)
print("=" * 50)

total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


---
## 7. Loss Functions

In [ ]:

# ============================================
# MULTI-TASK LOSS FUNCTION
# ============================================
_ts("Cell 15 - MultiTaskLoss")

class MultiTaskLoss(nn.Module):
    """
    Combined loss for multi-task learning.

    Loss = w_binding   * MSE(binding)
         + w_herg      * WeightedBCE(hERG)     [pos_weight for imbalance]
         + w_caco2     * MSE(caco2)             [regression, not classification]
         + w_clearance * MSE(clearance)
         + w_physics   * physics_penalty
    """
    def __init__(self, config):
        super(MultiTaskLoss, self).__init__()

        self.w_binding   = config['w_binding']
        self.w_herg      = config['w_herg']
        self.w_caco2     = config['w_caco2']
        self.w_clearance = config['w_clearance']
        self.w_physics   = config['w_physics']

        self.mse = nn.MSELoss()

        # Weighted BCE to handle hERG class imbalance (~14.9% positive)
        pos_w = torch.tensor([config.get('herg_pos_weight', 5.7)])
        self.bce_herg = nn.BCELoss(weight=None)      # per-sample weighting below
        self.herg_pos_weight = pos_w

    def _weighted_bce(self, preds, targets):
        """BCE with pos_weight for class imbalance."""
        eps = 1e-7
        pos_w = self.herg_pos_weight.to(preds.device)
        loss = -(pos_w * targets * torch.log(preds + eps)
                 + (1 - targets) * torch.log(1 - preds + eps))
        return loss.mean()

    def forward(self, predictions, targets, task):
        if task == 'binding':
            return self.w_binding * self.mse(predictions, targets)
        elif task == 'herg':
            return self.w_herg * self._weighted_bce(predictions, targets)
        elif task == 'caco2':
            return self.w_caco2 * self.mse(predictions, targets)
        elif task == 'clearance':
            return self.w_clearance * self.mse(predictions, targets)
        else:
            raise ValueError(f"Unknown task: {task}")

    def physics_penalty(self, pk_curve):
        neg_penalty       = torch.relu(-pk_curve).mean()
        dC                = pk_curve[1:] - pk_curve[:-1]
        increasing_penalty = torch.relu(dC).mean()
        return self.w_physics * (neg_penalty + increasing_penalty)

criterion = MultiTaskLoss(config)
print("MultiTaskLoss defined (weighted BCE for hERG, MSE for caco2 regression).")


---
## 8. Training Pipeline

In [ ]:

# ============================================
# TRAINING FUNCTION  (interleaved task sampling)
# ============================================
_ts("Cell 16 - Train Epoch Fn")

def train_epoch(model, data_loaders, optimizer, criterion, device, grad_clip=1.0):
    """
    Train one epoch with interleaved task sampling at the MINIMUM task size.
    Each epoch = min(task_batches) steps; larger tasks subsample their data.
    This avoids cycling small tasks (caco2 only has 10 batches) many times.
    DataLoader shuffle=True ensures different samples each epoch.
    """
    model.train()
    task_losses = {task: [] for task in data_loaders.keys()}

    # Use min batch count so no task repeats within an epoch
    n_steps = min(len(loader) for loader in data_loaders.values())

    # Fresh iterators (loaders have shuffle=True so order differs each epoch)
    iters = {task: iter(loader) for task, loader in data_loaders.items()}

    for _step in range(n_steps):
        for task in data_loaders.keys():
            try:
                batch = next(iters[task])
            except StopIteration:
                iters[task] = iter(data_loaders[task])
                batch = next(iters[task])

            X = batch['X'].to(device)
            y = batch['y'].to(device)

            optimizer.zero_grad()
            pred = model(X, task=task)
            loss = criterion(pred, y, task)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

            task_losses[task].append(loss.item())

    avg = {task: np.mean(losses) for task, losses in task_losses.items()}
    avg['total'] = sum(avg.values())
    return avg

print("Training function defined (min-step interleaving, gradient clipping).")


In [ ]:

# ============================================
# VALIDATION FUNCTION
# ============================================
_ts("Cell 17 - Validate Fn")

REGRESSION_TASKS     = {'binding', 'clearance', 'caco2'}   # MSE/R² metrics
CLASSIFICATION_TASKS = {'herg'}                             # AUROC/accuracy metrics

def validate(model, data_loaders, criterion, device):
    """
    Validate model on all tasks.
    Returns dict of per-task losses and metrics.
    """
    model.eval()
    results = {}

    with torch.no_grad():
        for task, loader in data_loaders.items():
            all_preds, all_targets, task_loss = [], [], []

            for batch in loader:
                X = batch['X'].to(device)
                y = batch['y'].to(device)

                pred = model(X, task=task)
                loss = criterion(pred, y, task)

                task_loss.append(loss.item())
                all_preds.extend(pred.cpu().numpy().flatten())
                all_targets.extend(y.cpu().numpy().flatten())

            preds   = np.array(all_preds)
            targets = np.array(all_targets)

            if task in REGRESSION_TASKS:
                results[task] = {
                    'loss': np.mean(task_loss),
                    'rmse': np.sqrt(mean_squared_error(targets, preds)),
                    'r2':   r2_score(targets, preds),
                    'mae':  mean_absolute_error(targets, preds),
                }
            else:  # classification
                binary_preds = (preds > 0.5).astype(int)
                results[task] = {
                    'loss':     np.mean(task_loss),
                    'auroc':    roc_auc_score(targets, preds) if len(np.unique(targets)) > 1 else 0.5,
                    'accuracy': accuracy_score(targets, binary_preds),
                    'f1':       f1_score(targets, binary_preds, zero_division=0),
                }

    return results

print("Validation function defined (caco2 now treated as regression).")


In [ ]:

# ============================================
# FULL TRAINING LOOP
# ============================================
_ts("Cell 18 - Train Model Fn")

def train_model(model, train_loaders, val_loaders, config, device):
    """Full training loop with early stopping."""
    optimizer = optim.Adam(
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=8
    )

    criterion = MultiTaskLoss(config)
    grad_clip  = config.get('grad_clip', 1.0)

    history = {'train_loss': [], 'val_loss': [], 'val_metrics': []}
    best_val_loss = float('inf')
    patience_counter = 0

    print(f"Starting training for up to {config['epochs']} epochs "
          f"(patience={config['patience']})...")
    print("=" * 65)

    for epoch in range(config['epochs']):
        train_losses = train_epoch(model, train_loaders, optimizer, criterion,
                                   device, grad_clip=grad_clip)
        val_results  = validate(model, val_loaders, criterion, device)
        val_loss     = sum(r['loss'] for r in val_results.values())

        scheduler.step(val_loss)

        history['train_loss'].append(train_losses['total'])
        history['val_loss'].append(val_loss)
        history['val_metrics'].append(val_results)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{config['epochs']}")
            print(f"  Train Loss: {train_losses['total']:.4f}  "
                  f"Val Loss: {val_loss:.4f}")
            for task, metrics in val_results.items():
                if 'r2' in metrics:
                    print(f"    {task:<10}: R\u00b2={metrics['r2']:.3f}, "
                          f"RMSE={metrics['rmse']:.3f}")
                else:
                    print(f"    {task:<10}: AUROC={metrics['auroc']:.3f}, "
                          f"Acc={metrics['accuracy']:.3f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model.pt')
        else:
            patience_counter += 1
            if patience_counter >= config['patience']:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break

    print(f"\nTraining complete! Best val loss: {best_val_loss:.4f}")
    model.load_state_dict(torch.load('best_model.pt', weights_only=True))
    return model, history

print("Training loop defined.")


---
## 9. Visualization Functions

In [ ]:

# ============================================
# TRAINING VISUALIZATION
# ============================================
_ts("Cell 19 - Plot History Fn")

REGRESSION_TASKS     = {'binding', 'clearance', 'caco2'}
CLASSIFICATION_TASKS = {'herg'}

def plot_training_history(history):
    """Plot training/validation loss and per-task metric curves."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Loss curves ──────────────────────────────────────────────
    ax = axes[0]
    ax.plot(history['train_loss'], label='Train Loss', color='steelblue')
    ax.plot(history['val_loss'],   label='Val Loss',   color='darkorange')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training & Validation Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # ── Per-task metrics ─────────────────────────────────────────
    ax = axes[1]
    tasks = list(history['val_metrics'][0].keys())
    colors = plt.cm.tab10.colors
    for i, task in enumerate(tasks):
        metric_name = 'r2' if task in REGRESSION_TASKS else 'auroc'
        values = [m[task][metric_name] for m in history['val_metrics']]
        ax.plot(values, label=f'{task} ({metric_name.upper()})', color=colors[i])

    ax.set_xlabel('Epoch')
    ax.set_ylabel('Score')
    ax.set_title('Task Performance Over Time')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Final metrics table ──────────────────────────────────────
    final = history['val_metrics'][-1]
    print("\nFinal Validation Metrics:")
    print(f"{'Task':<12} {'Metric':<8} {'Value':>8}  {'Target':>8}")
    print("-" * 42)
    targets_map = {
        'binding':   ('R²',   '>0.60',  'r2'),
        'herg':      ('AUROC','>0.80',  'auroc'),
        'caco2':     ('R²',   '>0.60',  'r2'),
        'clearance': ('RMSE', '<1.0',   'rmse'),
    }
    for task, (metric_label, target, key) in targets_map.items():
        val = final[task][key]
        print(f"  {task:<10} {metric_label:<8} {val:>8.3f}  {target:>8}")

print("Visualization functions defined.")


In [ ]:

# ============================================
# PK CURVE VISUALIZATION
# ============================================
_ts("Cell 20 - Plot PK Curves Fn")

def plot_pk_curves(model, sample_features, device, t_max=24, n_points=100):
    """
    Plot predicted PK concentration-time curves.
    
    Parameters:
    -----------
    model : MultiTaskPKPDModel
    sample_features : tensor
        Molecular features for compounds to plot
    device : torch device
    t_max : float
        Maximum time (hours)
    n_points : int
        Number of time points
    """
    model.eval()
    
    # Time span
    t_span = torch.linspace(0, t_max, n_points).to(device)
    
    # Predict PK curves
    X = torch.FloatTensor(sample_features).to(device)
    with torch.no_grad():
        C_t = model.predict_pk_curve(X, t_span, C0=1.0)
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    
    t_np = t_span.cpu().numpy()
    for i in range(min(5, X.shape[0])):
        C_np = C_t[:, i, 0].cpu().numpy()
        ax.plot(t_np, C_np, label=f'Compound {i+1}')
    
    ax.set_xlabel('Time (hours)')
    ax.set_ylabel('Concentration (normalized)')
    ax.set_title('Predicted PK Concentration-Time Curves (Neural ODE)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, t_max)
    ax.set_ylim(0, 1.1)
    
    plt.tight_layout()
    plt.savefig('pk_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("PK curves saved to 'pk_curves.png'")

print("PK curve visualization defined.")


---
## 10. Next Steps

### To Complete Phase 3:

1. **Data Preparation** (Cell 8-9)
   - Inspect actual column names in loaded datasets
   - Update feature/target column names
   - Create train/val/test splits
   - Build DataLoaders

2. **Model Training**
   - Run training loop
   - Monitor convergence
   - Tune hyperparameters

3. **Evaluation**
   - Compute final test metrics
   - Compare against baseline/targets
   - Generate PK curves

4. **Documentation**
   - Save results
   - Update PROJECT_SUMMARY.md

In [ ]:

# ============================================
# RUN TRAINING
# ============================================
_ts("Cell 21 - Training Run")

import time
t0 = time.time()

model, history = train_model(
    model,
    train_loaders,
    val_loaders,
    config,
    device
)

elapsed = time.time() - t0
print(f"\nTotal training time: {elapsed:.1f}s")
log_elapsed("Training complete")

# ---- Visualise ----
plot_training_history(history)


In [ ]:

# ============================================
# NEURAL ODE PK CURVE DEMO
# ============================================
_ts("Cell 22 - Results & Summary")

# Sample 5 Caco-2 test molecules for PK curve generation
sample_X = torch.FloatTensor(
    feat_scalers['clearance'].transform(X_clearance[:5])
)

print("Generating Neural ODE PK concentration-time curves...")
plot_pk_curves(model, sample_X.numpy(), device, t_max=24, n_points=100)

# ============================================
# PHASE 3 STATUS SUMMARY
# ============================================

print("\n" + "=" * 65)
print("PHASE 3 STATUS SUMMARY")
print("=" * 65)
print("\n✅ COMPLETED:")
print("  - Multi-task Neural ODE architecture built (44,710 params)")
print("  - 4-task pipeline: binding, hERG, Caco-2, clearance")
print("  - 68-dim features: 4 physico + 64 ECFP4 Morgan fingerprints")
print("  - LayerNorm encoder (BatchNorm-safe for multi-task training)")
print("  - Normalized targets (mean=0, std=1) for regression tasks")
print("  - Weighted BCE for hERG class imbalance (pos_weight=2.5)")
print("  - Neural ODE PK curve generation via torchdiffeq")
print("  - End-to-end training + early stopping + LR scheduling")

print("\n⚠️  PARTIAL:")
print("  - Clearance RMSE ≈ 0.97 (target <1.0 — borderline)")
print("  - All tasks near baseline (mean-prediction) performance")

print("\n❌ NOT YET MET:")
final = history['val_metrics'][-1]
targets_meta = [
    ('binding',   'R²',   '>0.60', 'r2'),
    ('herg',      'AUROC','>0.80', 'auroc'),
    ('caco2',     'R²',   '>0.60', 'r2'),
    ('clearance', 'RMSE', '<1.0',  'rmse'),
]
for task, metric, target, key in targets_meta:
    val = final[task][key]
    status = "✓" if (
        (metric == 'RMSE' and val < 1.0) or
        (metric != 'RMSE' and val > 0.6)
    ) else "✗"
    print(f"  {status} {task:<10} {metric:<6} = {val:.3f}  (target {target})")

print("\n📋 NEXT STEPS TO HIT TARGETS:")
print("  1. Increase Morgan fingerprints: 64 → 1024 bits (reduce collisions)")
print("  2. Add GNN molecular encoder for richer structural features")
print("  3. Task-specific encoders for binding (no SMILES available)")
print("  4. More training data via data augmentation or larger TDC splits")
print("=" * 65)
log_elapsed("Session complete")


In [ ]:

# =============================================================================
# CELL: Execution Timeline Summary
# PURPOSE: Display full execution order, timestamps, and elapsed times
# RUN: After all cells have been executed to verify sequential order
# =============================================================================

# Self-heal if this cell is run before the logger cell
if '_execution_log' not in globals() or not isinstance(_execution_log, dict):
    _execution_log = {}
    print("ℹ️  Logger state not found. Initialized empty _execution_log.")
    print("   Run cell 4 first for full timeline capture.")

if 'SESSION_START' not in globals():
    import datetime
    SESSION_START = datetime.datetime.now()

print(f"\n{'='*65}")
print("📋  PHASE 3 — EXECUTION TIMELINE SUMMARY")
print(f"{'='*65}")

if not _execution_log:
    print("  No cells logged yet — run all cells first.")
else:
    entries = list(_execution_log.items())
    start_time = entries[0][1]
    print(f"  {'#':<4}  {'Time':<10}  {'Elapsed':<12}  Cell")
    print(f"  {'-'*4}  {'-'*10}  {'-'*12}  {'-'*38}")
    for i, (name, t) in enumerate(entries, 1):
        elapsed = "(start)" if i == 1 else f"+{(t - start_time).total_seconds():.1f}s"
        print(f"  {i:<4}  {t.strftime('%H:%M:%S'):<10}  {elapsed:<12}  {name}")
    if len(entries) > 1:
        total = (entries[-1][1] - start_time).total_seconds()
        mins, secs = divmod(int(total), 60)
        print(f"\n  ✅ Total logged time : {mins}m {secs:02d}s  ({len(entries)} cells)")
        # Check for sequential execution order
        nums = []
        for name, _ in entries:
            try:
                nums.append(int(name.split("Cell")[1].split(" ")[1]))
            except Exception:
                pass
        if nums == sorted(nums):
            print("  ✅ Cells executed in sequential order")
        else:
            print("  ⚠️  Cells were NOT executed in sequential order!")
            print(f"     Detected order: {nums}")

print(f"{'='*65}")
